# 02 - Web Scraping Falcon 9 Launch Records from Wikipedia
Uses `requests` and `BeautifulSoup` to extract the launch tables and build a DataFrame.

In [ ]:
import requests, re, unicodedata, pandas as pd
from bs4 import BeautifulSoup
static_url='https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922'
response=requests.get(static_url)
soup=BeautifulSoup(response.text,'html.parser')
print(soup.title.string)

In [ ]:
html_tables=soup.find_all('table')
first_launch_table=html_tables[2]
column_names=[]
for th in first_launch_table.find_all('th'):
    name=' '.join(th.stripped_strings)
    if name and not name.isdigit(): column_names.append(name)
print(column_names)

In [ ]:
launch_dict=dict.fromkeys(column_names)
for k in ['Flight No.','Launch site','Payload','Payload mass','Orbit','Customer','Launch outcome','Version Booster','Booster landing','Date','Time']:
    launch_dict[k]=[]
extracted=0
for table in soup.find_all('table','wikitable plainrowheaders collapsible'):
    for rows in table.find_all('tr'):
        if rows.th and rows.th.string and rows.th.string.strip().isdigit():
            cells=[c.get_text(' ',strip=True) for c in rows.find_all('td')]
            if len(cells)>=8:
                launch_dict['Flight No.'].append(rows.th.get_text(strip=True))
                launch_dict['Date'].append(cells[0]);launch_dict['Time'].append(cells[1])
                launch_dict['Version Booster'].append(cells[2]);launch_dict['Launch site'].append(cells[3])
                launch_dict['Payload'].append(cells[4]);launch_dict['Payload mass'].append(cells[5])
                launch_dict['Orbit'].append(cells[6]);launch_dict['Customer'].append(cells[7])
                launch_dict['Launch outcome'].append(cells[8] if len(cells)>8 else '')
                launch_dict['Booster landing'].append(cells[9] if len(cells)>9 else '')
                extracted+=1
print('rows extracted:',extracted)
# NOTE: Wikipedia cell layout can change; inspect rows and adjust indices if needed.

In [ ]:
df=pd.DataFrame({k:v for k,v in launch_dict.items() if v})
df.to_csv('spacex_web_scraped.csv',index=False)
df.head()